# Two-kernel distributed delay via parallel linear chains

This notebook extends the MacDonald-style linear chain trick to a sum of two Erlang kernels,
$$G_{a_k}^p(u) = \frac{a_k^{p+1} u^p}{p!} e^{-a_k u}, \qquad k=1,2,$$
and solves the distributed-delay problem
$$\dot{x}(t) = \frac{-\tanh(\beta z(t)) - x(t)}{\tau}, \qquad z(t) = z_1(t) + z_2(t),$$
with
$$z_k(t) = \int_0^\infty G_{a_k}^p(u) x(t-u)\,du, \qquad k=1,2.$$
For each kernel, the delay term is exactly reproduced by its own linear chain
$$\dot{y}_0^{(k)} = a_k(x-y_0^{(k)}), \qquad \dot{y}_j^{(k)} = a_k(y_{j-1}^{(k)}-y_j^{(k)}), \quad j=1,\dots,p,$$
so that $z_k(t)=y_p^{(k)}(t)$ and $z(t)=y_p^{(1)}(t)+y_p^{(2)}(t)$. The resulting ODE system is integrated with `scipy.integrate.solve_ivp`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial
from scipy.integrate import solve_ivp
from scipy.special import gammaincc
from IPython.display import Audio

In [ ]:
def gamma_kernel(u, a, p):
    u = np.asarray(u)
    return (a ** (p + 1) / factorial(p)) * (u ** p) * np.exp(-a * u)


def delayed_rhs(t, state, tau, beta, a1, a2, p):
    del t
    chain_length = p + 1
    x = state[0]
    chain1 = state[1:1 + chain_length]
    chain2 = state[1 + chain_length:]
    z1 = chain1[-1]
    z2 = chain2[-1]
    z = z1 + z2

    dx = (-np.tanh(beta * z) - x) / tau

    dchain1 = np.empty_like(chain1)
    dchain1[0] = a1 * (x - chain1[0])
    dchain1[1:] = a1 * (chain1[:-1] - chain1[1:])

    dchain2 = np.empty_like(chain2)
    dchain2[0] = a2 * (x - chain2[0])
    dchain2[1:] = a2 * (chain2[:-1] - chain2[1:])

    return np.concatenate(([dx], dchain1, dchain2))


def solve_two_chain(tau, beta, mean_delay1, mean_delay2, p, x_history, t_eval):
    a1 = (p + 1) / mean_delay1
    a2 = (p + 1) / mean_delay2
    initial_state = np.full(2 * p + 3, x_history, dtype=float)

    sol = solve_ivp(
        delayed_rhs,
        (t_eval[0], t_eval[-1]),
        initial_state,
        t_eval=t_eval,
        args=(tau, beta, a1, a2, p),
        rtol=1e-8,
        atol=1e-10,
    )
    if not sol.success:
        raise RuntimeError(sol.message)

    chain_length = p + 1
    x = sol.y[0]
    z1 = sol.y[chain_length]
    z2 = sol.y[2 * chain_length]
    z = z1 + z2
    return a1, a2, x, z1, z2, z, sol


def convolution_reference_two(x, t_eval, x_history, a1, a2, p):
    dt = t_eval[1] - t_eval[0]
    kernel1 = gamma_kernel(t_eval, a1, p)
    kernel2 = gamma_kernel(t_eval, a2, p)

    causal1 = dt * np.convolve(x, kernel1, mode='full')[: len(t_eval)]
    causal2 = dt * np.convolve(x, kernel2, mode='full')[: len(t_eval)]
    history1 = x_history * gammaincc(p + 1, a1 * t_eval)
    history2 = x_history * gammaincc(p + 1, a2 * t_eval)

    z1_ref = causal1 + history1
    z2_ref = causal2 + history2
    return z1_ref, z2_ref, z1_ref + z2_ref

In [ ]:
tau = 0.9
beta = -2.1
mean_delay1 = 3.0
mean_delay2 = 7.0
p = 3
x_history = 0.2

t_eval = np.linspace(0.0, 100.0, 8001)
a1, a2, x, z1_chain, z2_chain, z_chain, sol = solve_two_chain(
    tau, beta, mean_delay1, mean_delay2, p, x_history, t_eval
)
z1_ref, z2_ref, z_ref = convolution_reference_two(x, t_eval, x_history, a1, a2, p)
max_error = np.max(np.abs(z_chain - z_ref))

print(f'order p = {p}')
print(f'tau = {tau:.3f}')
print(f'beta = {beta:.3f}')
print(f'a1 = {a1:.3f}  (mean delay 1 = {(p + 1) / a1:.3f})')
print(f'a2 = {a2:.3f}  (mean delay 2 = {(p + 1) / a2:.3f})')
print(f'max |z_chain - z_ref| = {max_error:.3e}')

In [ ]:
u = np.linspace(0.0, 4.0 * max(mean_delay1, mean_delay2), 1000)
kernel1 = gamma_kernel(u, a1, p)
kernel2 = gamma_kernel(u, a2, p)

fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=False)

axes[0].plot(u, kernel1, label='G₁(u)', color='tab:blue')
axes[0].plot(u, kernel2, label='G₂(u)', color='tab:orange')
axes[0].set_title('Two Erlang kernels')
axes[0].set_xlabel('u')
axes[0].set_ylabel('G(u)')
axes[0].legend()

axes[1].plot(t_eval, x, label='x(t)')
axes[1].plot(t_eval, z_chain, '--', label='z(t) = z₁ + z₂')
axes[1].set_title('State and total delayed variable')
axes[1].set_xlabel('t')
axes[1].legend()

axes[2].plot(t_eval, z1_chain, label='z₁(t) = y_p^(1)(t)', color='tab:blue')
axes[2].plot(t_eval, z2_chain, label='z₂(t) = y_p^(2)(t)', color='tab:orange')
axes[2].set_title('Delay components')
axes[2].set_xlabel('t')
axes[2].legend()

axes[3].plot(t_eval, z_chain - z_ref, color='tab:red')
axes[3].set_title('Two-chain error against direct convolution sum')
axes[3].set_xlabel('t')
axes[3].set_ylabel('z_chain - z_ref')

plt.tight_layout()
plt.show()

Audio(x, rate=8000)